In [0]:
%restart_python
%pip install boto3
import boto3
import os
from botocore.exceptions import NoCredentialsError
import datetime
import sys
sys.path.insert(0, '/Workspace/Shared')
import etl_helpers
from pyspark.sql.functions import lit, col

tablename = "factrequestextensions"
runcycleid = etl_helpers.start_run_cycle(tablename)

os.makedirs("/dbfs/foi/dataload", exist_ok=True)  # make sure directory exists

try:
    df_lastrun = spark.sql(f"SELECT runcyclestartat as createddate FROM dimruncycle WHERE packagename = \"{tablename}\" AND success = 't' ORDER BY runcycleid DESC LIMIT 1")
    
    if df_lastrun.count() > 0:
        lastruntime = df_lastrun.first().createddate.strftime("%Y-%m-%d %H:%M:%S")
    else:
        lastruntime = "2019-01-01 00:00:00"
    print(lastruntime)

    query = f"""
        select
            ext.foirequestextensionid,
            mr.axisrequestid as visualrequestfilenumber,
            er.reason,
            er.extensiontype,
            es.name as extensionstatus,
            ext.foiministryrequest_id as foiministryrequestid,
            ext.extendedduedays as extendednoofdays,

            ext.extendedduedate,
            mr.duedate as currentduedate,
            ext.decisiondate,
            ext.approvednoofdays
        from
        (
            select *
            from foi_mod.foirequestextensions
            where isactive = true
            QUALIFY ROW_NUMBER() OVER (PARTITION BY foirequestextensionid ORDER BY version DESC) = 1
        ) ext
        join (
            SELECT
                foiministryrequestid,
                axisrequestid,
                duedate
            From foi_mod.foiministryrequests
            QUALIFY ROW_NUMBER() OVER (PARTITION BY foiministryrequestid ORDER BY version DESC) = 1
        ) mr on mr.foiministryrequestid = ext.foiministryrequest_id
        
        inner join foi_mod.extensionreasons er on er.extensionreasonid = ext.extensionreasonid
        inner join foi_mod.extensionstatuses es on es.extensionstatusid = ext.extensionstatusid
        WHERE ext.created_at > CAST('{lastruntime}' AS TIMESTAMP)
        """

    print(query)

    df = spark.sql(query)
    df.show()

    if df.count() == 0:
        raise Exception("no changes for today")

    df_mapped = df.selectExpr(
        "foirequestextensionid AS foirequestextensionid",
        f"{runcycleid} as runcycleid",
        "try_cast(visualrequestfilenumber AS STRING) AS visualrequestfilenumber",
        "try_cast(reason AS STRING) AS reason",
        "try_cast(extensiontype AS STRING) AS extensiontype",
        "try_cast(extensionstatus AS STRING) AS extensionstatus",
        "try_cast(foiministryrequestid AS INT) AS foiministryrequestid",
        "try_cast(extendednoofdays AS INT) AS extendednoofdays",
        "try_cast(extendedduedate AS DATE) AS extendedduedate",
        "try_cast(currentduedate AS DATE) AS currentduedate",
        "try_cast(decisiondate AS DATE) AS decisiondate",
        "try_cast(approvednoofdays AS INT) AS approvednoofdays",
        "'Y' as activeflag"
    )
    df_mapped.show()

    table_path = f"hive_metastore.default.{tablename}"

    # check if target table exist
    if spark.catalog.tableExists(table_path):
        from delta.tables import DeltaTable
        delta_table = DeltaTable.forName(spark, table_path)
        delta_table.alias("target").merge(
            df_mapped.alias("source"),
            "target.foirequestextensionid = source.foirequestextensionid"
        ).whenMatchedUpdate(
            condition = "target.activeflag = 'Y'",
            set = {
                "activeflag": lit("N"),
            }
        ).execute()

        print("Matched records deactivated.")

        df_mapped.write.format("delta").mode("append").saveAsTable(table_path)
    else:
        # If target table doesn't exist, create the table using the mapped dataframe
        print(f"Table {table_path} does not exist. Creating it now.")
        df_mapped.write.format("delta").mode("overwrite").saveAsTable(table_path)
    etl_helpers.end_run_cycle(runcycleid, 't', tablename)
except NoCredentialsError:
    print("Credentials not available")
    etl_helpers.end_run_cycle(runcycleid, 'f', tablename, "Credentials not available")
    raise Exception("notebook failed") from e
except Exception as e:
    if (str(e) == "no changes for today"):
        print("here")
        etl_helpers.end_run_cycle(runcycleid, 't', tablename)
    else:
        print(f"An error occurred: {e}")    
        etl_helpers.end_run_cycle(runcycleid, 'f', tablename, f"An error occurred: {e}")
        raise Exception("notebook failed") from e